In [1]:
from utils import DQN, ReplayBuffer, greedy_action, epsilon_greedy, update_target, loss

import torch # ML library 
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
import math
import numpy as np

import gym # provides various RL environments
import matplotlib.pyplot as plt

import os
import fnmatch
import logging 

In [2]:
# Parameters 
NUM_RUNS = 3
num_episodes = 300

Hyperparameters Sweep

In [3]:
# !pip install wandb -Uq

In [4]:
import wandb
wandb.login()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: y-harayama187 (yuignite). Use `wandb login --relogin` to force relogin


True

In [5]:
sweep_config = {
    'method': 'random'
    }

metric = {
    'name': 'mean_return',
    'goal': 'maximize'   
    }

sweep_config['metric'] = metric

In [6]:
parameters_dict = {
    'optimizer': {
        'values': ['adam', 'sgd']
        },
    'BATCH_SIZE': {
        'values': [16, 32, 64, 128]
        },
    'BUFFER_SIZE':{
        'values': [100, 1000, 10000, 100000]
        },
    'UPDATE_PERIOD':{
        'values': [1, 10, 15]
    },
    'NUMBER_HIDDEN_LAYERS':{
        'values': [0, 1, 2, 3]
    },
    'SIZE_HIDDEN_LAYER':{
        'values': [32, 64, 128, 256, 512]
    },
    }

sweep_config['parameters'] = parameters_dict

In [7]:
parameters_dict.update({
    'LEARNING_RATE': {
        'distribution': 'uniform',
        'min': 0,
        'max': 1},
    'EPSILON': {
        'distribution': 'uniform',
        'min': 0,
        'max': 1},
    })

import pprint
pprint.pprint(sweep_config)

{'method': 'random',
 'metric': {'goal': 'maximize', 'name': 'mean_return'},
 'parameters': {'BATCH_SIZE': {'values': [16, 32, 64, 128]},
                'BUFFER_SIZE': {'values': [100, 1000, 10000, 100000]},
                'EPSILON': {'distribution': 'uniform', 'max': 1, 'min': 0},
                'LEARNING_RATE': {'distribution': 'uniform',
                                  'max': 1,
                                  'min': 0},
                'NUMBER_HIDDEN_LAYERS': {'values': [0, 1, 2, 3]},
                'SIZE_HIDDEN_LAYER': {'values': [32, 64, 128, 256, 512]},
                'UPDATE_PERIOD': {'values': [1, 10, 15]},
                'optimizer': {'values': ['adam', 'sgd']}}}


In [8]:
sweep_id = wandb.sweep(sweep_config, project="pytorch-hyperpara-sweeps")

Create sweep with ID: 1ifoei8l
Sweep URL: https://wandb.ai/yuignite/pytorch-hyperpara-sweeps/sweeps/1ifoei8l


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def train(config=None):
    # Initialize a new wandb run
    with wandb.init(config=config):
        # If called by wandb.agent, as below,
        # this config will be set by Sweep Controller
        config = wandb.config
        

        buffer_size = config.BUFFER_SIZE
        batch_size = config.BATCH_SIZE
        update_period = config.UPDATE_PERIOD
        size_inner_layer = config.SIZE_HIDDEN_LAYER
        number_inner_layers = config.NUMBER_HIDDEN_LAYERS
        lr = config.LEARNING_RATE
        epsilon = config.EPSILON

        #######
        runs_results = []

        env = gym.make('CartPole-v1')
        for run in range(NUM_RUNS):
            print(f"Starting run {run+1} of {NUM_RUNS}")

            architecture = [4, *[size_inner_layer]*number_inner_layers, 2]
            policy_net = DQN(architecture) # input = 4 and output = 2 (0 or 1) 
            target_net = DQN(architecture)
            update_target(target_net, policy_net)
            target_net.eval()
    
            optimizer = build_optimizer(policy_net, config.optimizer, lr)
            memory = ReplayBuffer(buffer_size) # 1 originally 

            steps_done = 0

            episode_durations = []

            for i_episode in range(num_episodes): # can adjust num_episodes 
                if (i_episode+1) % 50 == 0:
                    print("episode ", i_episode+1, "/", 300)

                observation, info = env.reset()
                state = torch.tensor(observation).float()

                done = False
                terminated = False
                t = 0
                # EPSILON = max(1/(i_episode+1), min_epsilon)
                while not (done or terminated):

                    # Select and perform an action
                    action = epsilon_greedy(epsilon, policy_net, state)
                    # action = greedy_action(policy_net, state) # can try without epsilon greedy 

                    # reward is a scalar, [reward] creates a one-element list, which is converted into a tensor
                    observation, reward, done, terminated, info = env.step(action)
                    reward = torch.tensor([reward])
                    action = torch.tensor([action])
                    next_state = torch.tensor(observation).reshape(-1).float()

                    # store in replay buffer
                    memory.push([state, action, next_state, reward, torch.tensor([done])])

                    # Move to the next state
                    state = next_state

                    # Perform one step of the optimization (on the policy network)
                    if not len(memory.buffer) < batch_size: # only proceed if enough data in buffer 
                        transitions = memory.sample(batch_size) # sample one transition from buffer
                        state_batch, action_batch, nextstate_batch, reward_batch, dones = (torch.stack(x) for x in zip(*transitions)) # and unpack
                        # Compute loss
                        mse_loss = loss(policy_net, target_net, state_batch, action_batch, reward_batch, nextstate_batch, dones)
                        # Optimize the model
                        optimizer.zero_grad() # from optimizer, reset gradient to prep for new step 
                        mse_loss.backward() # store gradient of the loss wrt each para 
                        optimizer.step() # carry out gradient dedscent step & update of para 
                    
                    if done or terminated:
                        episode_durations.append(t + 1)
                    t += 1
                # Update the target network, copying all weights and biases in DQN
                if i_episode % update_period == 0: 
                    update_target(target_net, policy_net)
            runs_results.append(episode_durations)
            print('log')
            wandb.log({"mean_return": np.mean(episode_durations[150:300])})
        print('Complete')


def build_optimizer(policy, optimizer, lr):
    if optimizer == "sgd":
        optimizer = optim.SGD(policy.parameters(),
                              lr) # removed momentum 
    elif optimizer == "adam":
        optimizer = optim.Adam(policy.parameters(),
                               lr)
    return optimizer            

In [10]:
wandb.agent(sweep_id, train, count=50)

wandb: Agent Starting Run: 316ysk2e with config:
wandb: 	BATCH_SIZE: 16
wandb: 	BUFFER_SIZE: 1000
wandb: 	EPSILON: 0.7395032749179927
wandb: 	LEARNING_RATE: 0.3831136383088322
wandb: 	NUMBER_HIDDEN_LAYERS: 1
wandb: 	SIZE_HIDDEN_LAYER: 32
wandb: 	UPDATE_PERIOD: 1
wandb: 	optimizer: adam
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


Starting run 1 of 3


C:\Users\yhara\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\gym\utils\passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


episode  50 / 300
episode  100 / 300
episode  150 / 300
episode  200 / 300
episode  250 / 300
episode  300 / 300
log
Starting run 2 of 3
episode  50 / 300
episode  100 / 300
episode  150 / 300
episode  200 / 300
episode  250 / 300
episode  300 / 300
log
Starting run 3 of 3
episode  50 / 300
episode  100 / 300
episode  150 / 300
episode  200 / 300
episode  250 / 300
episode  300 / 300
log
Complete


mean_return,█▂▁
mean_return,19.99333


...

Starting run 1 of 3
episode  50 / 300
episode  100 / 300
episode  150 / 300
episode  200 / 300
episode  250 / 300
episode  300 / 300
log
Starting run 2 of 3
episode  50 / 300
episode  100 / 300
episode  150 / 300
episode  200 / 300
episode  250 / 300
episode  300 / 300
log
Starting run 3 of 3
episode  50 / 300
episode  100 / 300
episode  150 / 300
episode  200 / 300
episode  250 / 300
episode  300 / 300
log
Complete


mean_return,▁█▃
mean_return,14.28667


wandb: Agent Starting Run: q74zqe04 with config:
wandb: 	BATCH_SIZE: 32
wandb: 	BUFFER_SIZE: 100
wandb: 	EPSILON: 0.7173017461592653
wandb: 	LEARNING_RATE: 0.4069342578589336
wandb: 	NUMBER_HIDDEN_LAYERS: 3
wandb: 	SIZE_HIDDEN_LAYER: 256
wandb: 	UPDATE_PERIOD: 15
wandb: 	optimizer: sgd
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


Starting run 1 of 3
episode  50 / 300
episode  100 / 300
episode  150 / 300
episode  200 / 300
episode  250 / 300
episode  300 / 300
log
Starting run 2 of 3
episode  50 / 300
episode  100 / 300
episode  150 / 300
episode  200 / 300
episode  250 / 300
episode  300 / 300
log
Starting run 3 of 3
episode  50 / 300
episode  100 / 300
episode  150 / 300
episode  200 / 300
episode  250 / 300
episode  300 / 300
log
Complete


mean_return,█▄▁
mean_return,17.39333
